# Notebook 01 — Exploring the Companies KG

Opening moves of the graph-navigation demo: connect to the live [Companies KG](https://demo.neo4jlabs.com)
and let the database describe itself.

Everything below goes through `src/neo4jev/neo4j_access.py`, and that library hardcodes no
label, index, relationship type or property name — all of it is read back from the live
graph (`CALL db.labels()`, `SHOW INDEXES`). This notebook does name a few labels to tour,
but only ones it first discovered in the graph, and it justifies the one index name it
picks. That schema-agnostic habit is what lets the navigator in the later notebooks work on
any graph.

The tour:

1. connect using the settings in `.env`
2. list the graph's labels
3. ask `detect_indexes()` which lookup modes (`exact` / `fulltext` / `vector`) each label supports
4. run one real `search_start_nodes()` lookup per detected mode

In [1]:
from contextlib import ExitStack

from neo4jev.neo4j_access import open_access

_stack = ExitStack()
access = _stack.enter_context(open_access())
print(f"Connected to database {access.database!r}")

Connected to database 'companies2'


## 1. Labels

`list_labels()` is a thin wrapper over `CALL db.labels()`. We report the database's labels
as-is rather than filtering them: `list_labels()` is a faithful view of the graph, and the
labels the demo actually navigates are picked later, on purpose.


In [2]:
labels = access.list_labels()
print(f"{len(labels)} labels")
for label in labels:
    print(" -", label)

15 labels
 - Article
 - CPCClass
 - Chunk
 - City
 - Country
 - IPCClass
 - IndustryCategory
 - Investment
 - NAICSCode
 - Organization
 - Patent
 - Person
 - Region
 - SECFiling
 - Technology


## 2. Which lookup modes does each label support?

* `exact` is always available — it is a plain substring/equality scan over every string
  property of the label, so it needs no index.
* `fulltext` and `vector` are reported **only** when the database actually has a matching
  index for that label, which is why `detect_indexes()` is the thing that decides which
  search modes the UI may offer.

For the vector indexes we also print the declared `vector.dimensions`, since that is what
an embedder has to produce.

In [3]:
labels_of_interest = ["Organization", "Person", "Article", "Chunk"]

indexes_by_label = {label: access.detect_indexes(label) for label in labels_of_interest}

for label, indexes in indexes_by_label.items():
    print(f"{label}: modes = {', '.join(indexes.modes)}")
    for ref in indexes.fulltext + indexes.vector:
        dimensions = f" dims={ref.dimensions}" if ref.dimensions else ""
        print(f"    {ref.kind:<8} {ref.name:<18} props={list(ref.properties)}{dimensions}")

Organization: modes = exact, fulltext
    FULLTEXT organization_fullName props=['fullName']
Person: modes = exact, fulltext
    FULLTEXT person_name        props=['name']
Article: modes = exact
Chunk: modes = exact, vector
    VECTOR   news_openai_small  props=['embedding_3_small', 'siteName', 'date', 'sentiment', 'language'] dims=1536


Notice how differently the four labels answer: `Article` gets nothing but `exact`,
`Organization` and `Person` add a fulltext index, and `Chunk` adds a vector one. The graph
therefore already answers "which searches can I offer for this label?" without any
configuration on our side.


In [4]:
def summarize(nodes):
    """One line per hit: the label, a readable caption, the element id and the score.

    The caption is a notebook-local convenience (a `name`/`fullName`/`title` property when
    the node has one, otherwise its longest string property) — `neo4j_access` itself
    assumes no property name at all.
    """
    for node in nodes:
        props = node.props
        caption = next(
            (str(props[key]) for key in ("name", "fullName", "title") if props.get(key)),
            "",
        )
        if not caption:
            strings = [v for v in props.values() if isinstance(v, str) and v.strip()]
            caption = max(strings, key=len) if strings else ""
        score = "" if node.score is None else f"  score={node.score:.3f}"
        print(f"  [{'/'.join(node.labels)}] {caption[:60]!r}  {node.element_id}{score}")


### 3a. `exact`

No index involved: the query is a case-insensitive `CONTAINS` test over *every* string
property of each `Organization`, not just its name. That is why organizations whose
`description` merely mentions Apple show up alongside the ones actually called `Apple`.
Whole-property equality is then used only for ranking, so the three organizations whose
`name` *is* `Apple` sort above the rest of the match set.


In [5]:
exact_hits = access.search_start_nodes("Organization", "Apple", "exact", limit=5)
print(f"exact match for 'Apple' -> {len(exact_hits)} hit(s)")
summarize(exact_hits)

exact match for 'Apple' -> 5 hit(s)
  [Organization] 'Apple'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:48007
  [Organization] 'APPLE'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:48009
  [Organization] 'Apple'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:48030
  [Organization] 'DoubleClick'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:11528
  [Organization] 'WSTM'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:13689


### 3b. `fulltext`

`detect_indexes("Organization")` found the `organization_fullName` fulltext index (on the
`fullName` property), so this mode is offered. The query text is Lucene-escaped by
`search_start_nodes()` before it reaches `db.index.fulltext.queryNodes`, so user input is
always treated as literal text.

Note which nodes this index reaches: `Apple Music`, `Apple Ads` and `Apple Inc.`. The index
is over the *full* name, so the shorter `name` property — `Apple`, on three distinct
organizations — is not what is being matched. Those three hits all tie on score, and the
relative order of tied hits is not stable, so anything downstream (like notebook 02) must
pick from the returned list rather than trusting `limit=1`.


In [6]:
fulltext_hits = access.search_start_nodes("Organization", "Apple", "fulltext", limit=5)
print(f"fulltext match for 'Apple' -> {len(fulltext_hits)} hit(s)")
summarize(fulltext_hits)

fulltext match for 'Apple' -> 3 hit(s)
  [Organization] 'Apple Music'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:8044  score=4.223
  [Organization] 'Apple Ads'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:8615  score=4.223
  [Organization] 'Apple Inc.'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:4  score=4.223


### 3c. `vector`

`Chunk` carries one vector index, `news_openai_small`, over the `embedding_3_small`
property, declaring **1536** dimensions. `search_start_nodes()` reads that dimension out of
`SHOW INDEXES` and asks the embedder for exactly that length; the sample below passes
`index_name="news_openai_small"` explicitly to make the choice self-documenting. (A label
can carry several vector indexes over several dimensions — this one does not.)

A caveat worth stating plainly: this demo has no embedding-provider credentials, so the
default embedder is a deterministic hash-seeded pseudo-embedding (`default_embedder` in
`neo4j_access.py`). The call exercises the index and returns real chunks in a real order,
but the ranking carries **no semantic meaning**. Pass a real embedder to `open_access()`
to get meaningful neighbours.


In [7]:
vector_hits = access.search_start_nodes(
    "Chunk", "renewable energy", "vector", limit=5, index_name="news_openai_small"
)
print(
    f"vector match for 'renewable energy' in index 'news_openai_small'"
    f" -> {len(vector_hits)} hit(s)"
)
summarize(vector_hits)


vector match for 'renewable energy' in index 'news_openai_small' -> 5 hit(s)
  [Chunk] 'は、中国東部における便数を適度に補完しながら、主に成都でのハブ建設を支え、中国南西部での同社航空機数を拡大することとなる'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:238093  score=0.543
  [Chunk] 'Whatever it takes for you to stay out jail Kodak……whatever i'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:210761  score=0.541
  [Chunk] '原标题:牧原股份:002714牧原股份调研活动信息20230603\n证券代码：002714 证券简称：牧原股份\n牧原食品'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:744796  score=0.538
  [Chunk] '代替旅客改期的情况。\n上述规定为什么在公开的退票规则中没有？\n工号63678客服说，旅客操作改期时都有相关提示，“如果旅'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:237959  score=0.537
  [Chunk] 'La compañía aérea Air France-KLM ha solicitado ayuda financi'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:749620  score=0.537


### 3d. Fulltext on `Person`

`Person` carries the graph's other fulltext index, `person_name` (over the `name`
property), so the same one-line lookup works on a second label with no per-label wiring.
Searching for `Tim Cook` returns the person himself and then the other `Cook`s the index
ranks underneath him.


In [8]:
person_fulltext_hits = access.search_start_nodes("Person", "Tim Cook", "fulltext", limit=5)
print(f"fulltext match for 'Tim Cook' -> {len(person_fulltext_hits)} hit(s)")
summarize(person_fulltext_hits)


fulltext match for 'Tim Cook' -> 5 hit(s)
  [Person] 'Tim Cook'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:504  score=6.319
  [Person] 'Scott Cook'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:1966  score=3.587
  [Person] 'Ian Cook'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:1707  score=3.587
  [Person] 'Christopher Cook'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:185909  score=3.587
  [Person] 'Terrance Cook'  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:7003  score=3.587


## 4. The detection is real, not cosmetic

Asking for a mode the graph cannot serve fails loudly instead of silently returning
nothing, which is what lets the Streamlit app build its mode picker straight from
`detect_indexes()`.

In [9]:
try:
    access.search_start_nodes("Article", "energy", "fulltext")
except ValueError as exc:
    print(f"Article supports only {indexes_by_label['Article'].modes} -> {exc}")

Article supports only ('exact',) -> No fulltext index available for label 'Article'


## Next

Notebook 02 (`02_navigator_dry_run.ipynb`) picks a start node with the lookup modes shown
here, fetches its outgoing relationships, and runs a single TypeSafe `Choice` + `Noul` hop
against it.

In [10]:
_stack.close()
print("Connection closed.")

Connection closed.
